# 5. YOLO detector (future work, not used in the current scope)
Fine-tunes a YOLO detection model on the scenes from notebook 04. Set `YOLO_FILE=yolo_best.pt` in `.env` so the API serves it (it then returns boxes and joins the labels left to right).

In [1]:
import os, sys
from pathlib import Path

# run everything from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))  # so `backend.ml.preprocessing` can be imported
print("project root:", ROOT)

project root: D:\Portfolio Projects\MODI Script Identification


In [2]:
# ---- settings ----
DATA = "data/processed/yolo_scenes/data.yaml"
WEIGHTS = "yolov8n.pt"      # or yolo11n.pt
EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = None               # e.g. 0 for GPU, "cpu"
SEED = 42
OUT = "models/yolo_best.pt"
REPORT_DIR = "reports/yolo"

In [ ]:
import json, shutil

from ultralytics import YOLO

model = YOLO(WEIGHTS)
model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
            project="runs/yolo", name="modi", exist_ok=True, patience=20, seed=SEED)

best = Path(model.trainer.save_dir) / "weights" / "best.pt"
Path(OUT).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, OUT)

New https://pypi.org/project/ultralytics/8.4.192 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.191  Python-3.14.3 torch-2.14.0+cpu CPU (Intel Core 5 120U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/yolo_scenes/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.

In [ ]:
m = YOLO(OUT).val(data=DATA, split="test", imgsz=IMGSZ, device=DEVICE)
metrics = {"map50": float(m.box.map50), "map50_95": float(m.box.map),
           "precision": float(m.box.mp), "recall": float(m.box.mr)}
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
(Path(REPORT_DIR) / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("test metrics:", metrics)
print(f"saved {OUT}")